In [1]:
!python --version

Python 3.10.18


In [2]:
import json
import random
import os

os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"
os.environ["VLLM_LOGGING_LEVEL"] = "ERROR"

from datasets import load_dataset
from langchain_community.llms import VLLM
from langchain_huggingface import HuggingFaceEmbeddings
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer

from rag_bench import baseline, data, evaluator, results

In [3]:
HIST_PRIVATE_QA_REPO_ID: str = "ai-forever/hist-rag-bench-private-qa"
HIST_PRIVATE_TEXTS_REPO_ID: str = "ai-forever/hist-rag-bench-private-texts"
RANDOM_SEED: int = 42
EMBEDDER_NAME: str = "ai-forever/FRIDA"
LLM_NAME: str = "bond005/meno-tiny-0.1"

In [4]:
LLM_PROMPT: str = """Проанализируйте заданный контекст и ответьте на вопрос пользователя на основе сведений, предоставленных в этом контексте.
Не давайте никаких объяснений и пояснений к своему ответу. Не пишите ничего лишнего. Не извиняйтесь, не стройте диалог. Выдавайте только ответ и ничего больше.
Отвечайте на русском языке.
Если в заданном контексте нет информации для ответа на вопрос пользователя, то ничего не придумывайте и просто откажитесь отвечать.
"""
print(LLM_PROMPT)

Проанализируйте заданный контекст и ответьте на вопрос пользователя на основе сведений, предоставленных в этом контексте.
Не давайте никаких объяснений и пояснений к своему ответу. Не пишите ничего лишнего. Не извиняйтесь, не стройте диалог. Выдавайте только ответ и ничего больше.
Отвечайте на русском языке.
Если в заданном контексте нет информации для ответа на вопрос пользователя, то ничего не придумывайте и просто откажитесь отвечать.



In [5]:
def get_private_qa_dataset(version):
    return load_dataset(HIST_PRIVATE_QA_REPO_ID, revision=version)


def get_private_texts_dataset(version):
    return load_dataset(HIST_PRIVATE_TEXTS_REPO_ID, revision=version)


def get_public_to_private_texts_mapping(version):
    private_texts_ds = get_private_texts_dataset(version)
    mapping = {}
    for item in private_texts_ds["train"]:
        mapping[item["public_id"]] = item["id"]
    return mapping

In [6]:
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.random.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_SEED)

In [7]:
#get public datasets (history ones)
texts_ds, questions_ds, version = data.get_datasets(is_hist=True)
print(f"version = {version}")

Latest texts version: 1.13.0
Latest questions version: 1.13.0
Loaded texts dataset with 560 texts
Loaded questions dataset with 600 questions
version = 1.13.0


In [8]:
#get private datasets (history ones)
qa_dataset = get_private_qa_dataset(version)
mapping = get_public_to_private_texts_mapping(version)

In [9]:
llm = VLLM(
    model=LLM_NAME,
    tensor_parallel_size=1,
    max_new_tokens=256,
    top_p=0.95,
    temperature=0.3,
    vllm_kwargs={
        "gpu_memory_utilization": 0.45,
        "max_num_batched_tokens": 8192,
        "max_model_len": 4096,
        "disable_log_stats": True,
        "seed": RANDOM_SEED
    }
)
tok = AutoTokenizer.from_pretrained(LLM_NAME)

[W114 17:13:45.400549202 ProcessGroupNCCL.cpp:981] Warning: TORCH_NCCL_AVOID_RECORD_STREAMS is the default now, this environment variable is thus deprecated. (function operator())


[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0


Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.32it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:00<00:00,  2.32it/s]
(EngineCore_DP0 pid=41758) 
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 67/67 [00:03<00:00, 20.93it/s]


In [10]:
embedding_model = HuggingFaceEmbeddings(
    model_name="ai-forever/FRIDA",
    model_kwargs={"trust_remote_code": True},
    encode_kwargs={"batch_size": 16, "prompt": "search_document: "},
    query_encode_kwargs={"prompt": "search_query: "}
)

In [11]:
retrieval = baseline.init_retriever(
    texts_ds,
    embedding_model,
    top_k=5,
    chunk_size=500,
    chunk_overlap=100,
)
generation = baseline.init_generation(retrieval, llm, tok, system_prompt=LLM_PROMPT)

Initializing retriever
> Creating vector store
> Vector store created for 560 documents
> Done for 57.21 seconds
Initializing generation chain
> Done


In [12]:
res = baseline.get_results(
    generation, questions_ds, write_logs=False
)

Calculating and preparing results


  0%|                                                                                                                | 0/600 [00:00<?, ?it/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                           | 0/1 [00:00<?, ?it/s, est. speed input: 0.…

  0%|▏                                                                                                       | 1/600 [00:00<04:03,  2.46it/s]

...

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████▊| 599/600 [02:37<00:00,  3.63it/s]

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0%|                                           | 0/1 [00:00<?, ?it/s, est. speed input: 0.…

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████| 600/600 [02:37<00:00,  3.81it/s]

Done for 157.68 seconds


In [13]:
#saving the baseline results

res_path = "./test.json"
results.save(res, res_path)

In [14]:
demo_public_id = 0
print(questions_ds["train"].filter(lambda it: it["id"] == demo_public_id)[0])
print('\n' + json.dumps(res[demo_public_id], ensure_ascii=False, indent=4))

{'id': 0, 'question': 'Каково атмосферное давление в Москве?'}

{
    "found_ids": [
        156,
        63,
        16,
        47,
        2
    ],
    "model_answer": "747 миллиметров ртутного столба"
}


In [15]:
#calculate metrics

evaluation_results = evaluator.evaluate_rag_results(res, qa_dataset, mapping)

In [16]:
#average metrics

res_agg = evaluation_results.average_metrics

df_retrieval = pd.DataFrame([res_agg["retrieval"]], index=["retrieval"]).apply(pd.to_numeric, errors="coerce")
df_generation = pd.DataFrame([res_agg["generation"]], index=["generation"]).apply(pd.to_numeric, errors="coerce")

df_retrieval = df_retrieval.round(4)
df_generation = df_generation.round(4)

In [17]:
df_retrieval

,hit_rate,mrr
retrieval,0.8112,0.8037


In [18]:
df_generation

,rouge1,rouge2,rougeL,em,substring_match
generation,0.4967,0.3092,0.476,0.2133,0.255
